# Step 3: Clean Geolocation Data

This step creates a clean zip-code-level geolocation table for matching customer and seller locations.

In [1]:
# Step 3: Clean geolocation data at zip-code-prefix level

from pathlib import Path
import pandas as pd
from IPython.display import display

# Set project directories
project_dir = Path("/Users/mac/Desktop/portfolio2_logistics_optimization")
data_raw_dir = project_dir / "data" / "raw"
data_processed_dir = project_dir / "data" / "processed"

# Load raw geolocation data
geo = pd.read_csv(data_raw_dir / "olist_geolocation_dataset.csv")

print("=== Raw Geolocation Data Shape ===")
print("geo:", geo.shape)

print("\n=== Raw Geolocation Columns ===")
print(list(geo.columns))

# Check missing values
print("\n=== Missing Values in Raw Geolocation Data ===")
geo_missing = geo.isna().sum().reset_index()
geo_missing.columns = ["column_name", "missing_count"]
geo_missing["missing_rate"] = geo_missing["missing_count"] / len(geo)
display(geo_missing)

# Create a clean zip-code-level geolocation table
# The same zip code prefix may appear multiple times, so we aggregate latitude and longitude by mean.
geo_clean = (
    geo.groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(
        latitude=("geolocation_lat", "mean"),
        longitude=("geolocation_lng", "mean"),
        city=("geolocation_city", "first"),
        state=("geolocation_state", "first")
    )
)

# Rename zip code column for easier merging later
geo_clean = geo_clean.rename(
    columns={"geolocation_zip_code_prefix": "zip_code_prefix"}
)

print("\n=== Clean Geolocation Data Shape ===")
print("geo_clean:", geo_clean.shape)

print("\n=== Sample Rows ===")
display(geo_clean.head())

print("\n=== State Distribution in Clean Geolocation Data ===")
state_summary = (
    geo_clean["state"]
    .value_counts()
    .reset_index()
)

state_summary.columns = ["state", "zip_code_count"]
display(state_summary.head(10))

# Save clean geolocation table
output_path = data_processed_dir / "geolocation_clean.csv"
geo_clean.to_csv(output_path, index=False)

print("\n=== Step 3 Final Result ===")
print("Saved clean geolocation table to:")
print(output_path)

=== Raw Geolocation Data Shape ===
geo: (1000163, 5)

=== Raw Geolocation Columns ===
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

=== Missing Values in Raw Geolocation Data ===


,column_name,missing_count,missing_rate
0,geolocation_zip_code_prefix,0,0.0
1,geolocation_lat,0,0.0
2,geolocation_lng,0,0.0
3,geolocation_city,0,0.0
4,geolocation_state,0,0.0



=== Clean Geolocation Data Shape ===
geo_clean: (19015, 5)

=== Sample Rows ===


,zip_code_prefix,latitude,longitude,city,state
0,1001,-23.550190,-46.634024,sao paulo,SP
1,1002,-23.548146,-46.634979,sao paulo,SP
2,1003,-23.548994,-46.635731,sao paulo,SP
3,1004,-23.549799,-46.634757,sao paulo,SP
4,1005,-23.549456,-46.636733,sao paulo,SP



=== State Distribution in Clean Geolocation Data ===


,state,zip_code_count
0,SP,6349
1,MG,1868
2,RJ,1390
3,RS,1131
4,PR,1046
5,BA,992
6,GO,773
7,SC,619
8,PE,596
9,CE,548



=== Step 3 Final Result ===
Saved clean geolocation table to:
/Users/mac/Desktop/portfolio2_logistics_optimization/data/processed/geolocation_clean.csv


# Step 4: Match Customer and Seller Coordinates

This step matches zip-code-level latitude and longitude to customer and seller locations.

In [2]:
# Step 4: Match customer and seller coordinates to logistics base table

from pathlib import Path
import pandas as pd
from IPython.display import display

# Set project directories
project_dir = Path("/Users/mac/Desktop/portfolio2_logistics_optimization")
data_processed_dir = project_dir / "data" / "processed"

# Load processed logistics base table and clean geolocation table
base_df = pd.read_csv(data_processed_dir / "order_logistics_base.csv")
geo_clean = pd.read_csv(data_processed_dir / "geolocation_clean.csv")

print("=== Input Table Shapes ===")
print("order_logistics_base:", base_df.shape)
print("geolocation_clean:", geo_clean.shape)

# Prepare customer geolocation table
customer_geo = geo_clean.rename(
    columns={
        "zip_code_prefix": "customer_zip_code_prefix",
        "latitude": "customer_latitude",
        "longitude": "customer_longitude",
        "city": "customer_geo_city",
        "state": "customer_geo_state"
    }
)

# Prepare seller geolocation table
seller_geo = geo_clean.rename(
    columns={
        "zip_code_prefix": "seller_zip_code_prefix",
        "latitude": "seller_latitude",
        "longitude": "seller_longitude",
        "city": "seller_geo_city",
        "state": "seller_geo_state"
    }
)

# Match customer coordinates
logistics_geo_df = base_df.merge(
    customer_geo,
    on="customer_zip_code_prefix",
    how="left"
)

# Match seller coordinates
logistics_geo_df = logistics_geo_df.merge(
    seller_geo,
    on="seller_zip_code_prefix",
    how="left"
)

print("\n=== Logistics Table With Coordinates Shape ===")
print("logistics_geo_df:", logistics_geo_df.shape)

# Check missing coordinate values
coordinate_columns = [
    "customer_latitude",
    "customer_longitude",
    "seller_latitude",
    "seller_longitude"
]

coordinate_missing = logistics_geo_df[coordinate_columns].isna().sum().reset_index()
coordinate_missing.columns = ["coordinate_column", "missing_count"]
coordinate_missing["missing_rate"] = coordinate_missing["missing_count"] / len(logistics_geo_df)

print("\n=== Missing Coordinate Check ===")
display(coordinate_missing)

# Show sample rows with coordinates
print("\n=== Sample Rows With Coordinates ===")
sample_columns = [
    "order_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state",
    "customer_latitude",
    "customer_longitude",
    "seller_zip_code_prefix",
    "seller_city",
    "seller_state",
    "seller_latitude",
    "seller_longitude",
    "freight_value"
]

display(logistics_geo_df[sample_columns].head())

# Save table with coordinates
output_path = data_processed_dir / "order_logistics_with_coordinates.csv"
logistics_geo_df.to_csv(output_path, index=False)

print("\n=== Step 4 Final Result ===")
print("Saved logistics table with customer and seller coordinates to:")
print(output_path)

=== Input Table Shapes ===
order_logistics_base: (110197, 13)
geolocation_clean: (19015, 5)

=== Logistics Table With Coordinates Shape ===
logistics_geo_df: (110197, 21)

=== Missing Coordinate Check ===


,coordinate_column,missing_count,missing_rate
0,customer_latitude,288,0.002614
1,customer_longitude,288,0.002614
2,seller_latitude,249,0.002260
3,seller_longitude,249,0.002260



=== Sample Rows With Coordinates ===


,order_id,customer_zip_code_prefix,customer_city,customer_state,customer_latitude,customer_longitude,seller_zip_code_prefix,seller_city,seller_state,seller_latitude,seller_longitude,freight_value
0,e481f51cbdc54678b7cc49136f2d6af7,3149,sao paulo,SP,-23.576983,-46.587161,9350,maua,SP,-23.680729,-46.444238,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,47813,barreiras,BA,-12.177924,-44.660711,31570,belo horizonte,SP,-19.807681,-43.980427,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,75265,vianopolis,GO,-16.745150,-48.514783,14840,guariba,SP,-21.363502,-48.229601,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,59296,sao goncalo do amarante,RN,-5.774190,-35.271143,31842,belo horizonte,MG,-19.837682,-43.924053,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,9195,santo andre,SP,-23.676370,-46.514627,8752,mogi das cruzes,SP,-23.543395,-46.262086,8.72



=== Step 4 Final Result ===
Saved logistics table with customer and seller coordinates to:
/Users/mac/Desktop/portfolio2_logistics_optimization/data/processed/order_logistics_with_coordinates.csv


# Step 5: Calculate Seller-Customer Distance

This step calculates the distance between seller locations and customer locations using latitude and longitude.

In [3]:
# Step 5: Calculate seller-customer distance using latitude and longitude

from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

# Set project directories
project_dir = Path("/Users/mac/Desktop/portfolio2_logistics_optimization")
data_processed_dir = project_dir / "data" / "processed"

# Load logistics table with coordinates
logistics_geo_df = pd.read_csv(data_processed_dir / "order_logistics_with_coordinates.csv")

print("=== Input Table Shape ===")
print("logistics_geo_df:", logistics_geo_df.shape)

# Define coordinate columns
coordinate_columns = [
    "customer_latitude",
    "customer_longitude",
    "seller_latitude",
    "seller_longitude"
]

# Remove rows with missing coordinates
rows_before = len(logistics_geo_df)

distance_df = logistics_geo_df.dropna(subset=coordinate_columns).copy()

rows_after = len(distance_df)
rows_removed = rows_before - rows_after
removed_rate = rows_removed / rows_before

print("\n=== Coordinate Cleaning Result ===")
print("Rows before removing missing coordinates:", rows_before)
print("Rows after removing missing coordinates:", rows_after)
print("Rows removed:", rows_removed)
print("Removed rate:", removed_rate)

# Define haversine distance function
# This calculates approximate distance between two latitude-longitude points on Earth.
def haversine_distance_km(lat1, lon1, lat2, lon2):
    earth_radius_km = 6371
    
    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)
    
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    
    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2) ** 2
    )
    
    c = 2 * np.arcsin(np.sqrt(a))
    
    return earth_radius_km * c

# Calculate seller-customer distance
distance_df["distance_km"] = haversine_distance_km(
    distance_df["seller_latitude"],
    distance_df["seller_longitude"],
    distance_df["customer_latitude"],
    distance_df["customer_longitude"]
)

print("\n=== Distance Summary ===")
distance_summary = distance_df["distance_km"].describe().reset_index()
distance_summary.columns = ["metric", "value"]
display(distance_summary)

print("\n=== Sample Rows With Distance ===")
sample_columns = [
    "order_id",
    "seller_city",
    "seller_state",
    "customer_city",
    "customer_state",
    "seller_latitude",
    "seller_longitude",
    "customer_latitude",
    "customer_longitude",
    "distance_km",
    "freight_value"
]

display(distance_df[sample_columns].head())

# Save processed table with distance
output_path = data_processed_dir / "order_logistics_with_distance.csv"
distance_df.to_csv(output_path, index=False)

print("\n=== Step 5 Final Result ===")
print("Saved logistics table with seller-customer distance to:")
print(output_path)

=== Input Table Shape ===
logistics_geo_df: (110197, 21)

=== Coordinate Cleaning Result ===
Rows before removing missing coordinates: 110197
Rows after removing missing coordinates: 109661
Rows removed: 536
Removed rate: 0.004864016261785711

=== Distance Summary ===


,metric,value
0,count,109661.000000
1,mean,596.233801
2,std,588.541210
3,min,0.000000
4,25%,185.205859
5,50%,431.857678
6,75%,791.627478
7,max,8677.911622



=== Sample Rows With Distance ===


,order_id,seller_city,seller_state,customer_city,customer_state,seller_latitude,seller_longitude,customer_latitude,customer_longitude,distance_km,freight_value
0,e481f51cbdc54678b7cc49136f2d6af7,maua,SP,sao paulo,SP,-23.680729,-46.444238,-23.576983,-46.587161,18.576110,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,belo horizonte,SP,barreiras,BA,-19.807681,-43.980427,-12.177924,-44.660711,851.495069,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,guariba,SP,vianopolis,GO,-21.363502,-48.229601,-16.745150,-48.514783,514.410666,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,belo horizonte,MG,sao goncalo do amarante,RN,-19.837682,-43.924053,-5.774190,-35.271143,1822.226336,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,mogi das cruzes,SP,santo andre,SP,-23.543395,-46.262086,-23.676370,-46.514627,29.676625,8.72



=== Step 5 Final Result ===
Saved logistics table with seller-customer distance to:
/Users/mac/Desktop/portfolio2_logistics_optimization/data/processed/order_logistics_with_distance.csv


# Step 6: Create Customer Demand Zones and Seller Fulfillment Centers

This step aggregates order-level logistics data into customer demand zones and seller fulfillment centers for the optimization model.

In [4]:
# Step 6: Create customer demand zones and seller fulfillment centers

from pathlib import Path
import pandas as pd
from IPython.display import display

# Set project directories
project_dir = Path("/Users/mac/Desktop/portfolio2_logistics_optimization")
data_processed_dir = project_dir / "data" / "processed"

# Load logistics table with distance
distance_df = pd.read_csv(data_processed_dir / "order_logistics_with_distance.csv")

print("=== Input Table Shape ===")
print("order_logistics_with_distance:", distance_df.shape)

# Create customer demand zones at city-state level
customer_zones = (
    distance_df
    .groupby(["customer_city", "customer_state"], as_index=False)
    .agg(
        demand=("order_id", "count"),
        customer_latitude=("customer_latitude", "mean"),
        customer_longitude=("customer_longitude", "mean"),
        avg_freight_value=("freight_value", "mean"),
        avg_actual_distance_km=("distance_km", "mean")
    )
)

# Create a clean customer zone ID
customer_zones["customer_zone_id"] = (
    customer_zones["customer_city"].str.replace(" ", "_", regex=False)
    + "_"
    + customer_zones["customer_state"]
)

# Reorder columns
customer_zones = customer_zones[
    [
        "customer_zone_id",
        "customer_city",
        "customer_state",
        "demand",
        "customer_latitude",
        "customer_longitude",
        "avg_freight_value",
        "avg_actual_distance_km"
    ]
]

# Keep top customer zones to make the optimization model manageable
top_n_customer_zones = 30

customer_zones_top = (
    customer_zones
    .sort_values("demand", ascending=False)
    .head(top_n_customer_zones)
    .reset_index(drop=True)
)

# Create seller fulfillment centers at seller zip-code level
seller_centers = (
    distance_df
    .groupby(["seller_zip_code_prefix", "seller_city", "seller_state"], as_index=False)
    .agg(
        historical_orders=("order_id", "count"),
        seller_latitude=("seller_latitude", "mean"),
        seller_longitude=("seller_longitude", "mean"),
        avg_freight_value=("freight_value", "mean"),
        avg_actual_distance_km=("distance_km", "mean")
    )
)

# Create a clean seller center ID
seller_centers["seller_center_id"] = (
    "FC_"
    + seller_centers["seller_zip_code_prefix"].astype(str)
    + "_"
    + seller_centers["seller_city"].str.replace(" ", "_", regex=False)
    + "_"
    + seller_centers["seller_state"]
)

# Reorder columns
seller_centers = seller_centers[
    [
        "seller_center_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state",
        "historical_orders",
        "seller_latitude",
        "seller_longitude",
        "avg_freight_value",
        "avg_actual_distance_km"
    ]
]

# Keep top seller centers to make the optimization model manageable
top_n_seller_centers = 12

seller_centers_top = (
    seller_centers
    .sort_values("historical_orders", ascending=False)
    .head(top_n_seller_centers)
    .reset_index(drop=True)
)

print("\n=== Customer Demand Zones ===")
print("All customer zones:", customer_zones.shape)
print("Selected top customer zones:", customer_zones_top.shape)
display(customer_zones_top)

print("\n=== Seller Fulfillment Centers ===")
print("All seller centers:", seller_centers.shape)
print("Selected top seller centers:", seller_centers_top.shape)
display(seller_centers_top)

print("\n=== Demand and Capacity Planning Check ===")
total_selected_demand = customer_zones_top["demand"].sum()
total_selected_historical_orders = seller_centers_top["historical_orders"].sum()

print("Total demand in selected customer zones:", total_selected_demand)
print("Total historical orders in selected seller centers:", total_selected_historical_orders)

# Save processed node tables
customer_output_path = data_processed_dir / "customer_demand_zones.csv"
seller_output_path = data_processed_dir / "seller_fulfillment_centers.csv"

customer_zones_top.to_csv(customer_output_path, index=False)
seller_centers_top.to_csv(seller_output_path, index=False)

print("\n=== Step 6 Final Result ===")
print("Saved customer demand zones to:")
print(customer_output_path)
print("Saved seller fulfillment centers to:")
print(seller_output_path)

=== Input Table Shape ===
order_logistics_with_distance: (109661, 22)

=== Customer Demand Zones ===
All customer zones: (4218, 8)
Selected top customer zones: (30, 8)


,customer_zone_id,customer_city,customer_state,demand,customer_latitude,customer_longitude,avg_freight_value,avg_actual_distance_km
0,sao_paulo_SP,sao paulo,SP,17359,-23.571981,-46.633876,14.275696,217.020084
1,rio_de_janeiro_RJ,rio de janeiro,RJ,7565,-22.923627,-43.321220,20.569699,468.031288
2,belo_horizonte_MG,belo horizonte,MG,3077,-19.909936,-43.956380,19.415499,521.709241
3,brasilia_DF,brasilia,DF,2157,-15.810745,-47.969518,21.108790,830.677176
4,curitiba_PR,curitiba,PR,1723,-25.451862,-49.274718,18.792101,404.235197
5,campinas_SP,campinas,SP,1622,-22.902106,-47.074802,14.954969,228.563150
6,porto_alegre_RS,porto alegre,RS,1567,-30.049836,-51.186022,20.671914,877.107002
7,salvador_BA,salvador,BA,1349,-12.959523,-38.458218,25.378614,1402.125961
8,guarulhos_SP,guarulhos,SP,1292,-23.444952,-46.498584,14.464799,223.451228
9,sao_bernardo_do_campo_SP,sao bernardo do campo,SP,1037,-23.706925,-46.564591,13.702893,205.925633



=== Seller Fulfillment Centers ===
All seller centers: (2209, 9)
Selected top seller centers: (12, 9)


,seller_center_id,seller_zip_code_prefix,seller_city,seller_state,historical_orders,seller_latitude,seller_longitude,avg_freight_value,avg_actual_distance_km
0,FC_14940_ibitinga_SP,14940,ibitinga,SP,7595,-21.757321,-48.829744,17.772378,539.555918
1,FC_5849_sao_paulo_SP,5849,sao paulo,SP,2004,-23.652366,-46.755753,13.733613,600.103332
2,FC_15025_sao_jose_do_rio_preto_SP,15025,sao jose do rio preto,SP,1998,-20.806707,-49.389165,18.312107,691.092432
3,FC_9015_santo_andre_SP,9015,santo andre,SP,1710,-23.659364,-46.523183,14.450211,609.935829
4,FC_13405_piracicaba_SP,13405,piracicaba,SP,1562,-22.708702,-47.664701,16.162823,402.102118
5,FC_4782_sao_paulo_SP,4782,sao paulo,SP,1485,-23.691013,-46.703810,16.803589,488.181686
6,FC_8577_itaquaquecetuba_SP,8577,itaquaquecetuba,SP,1441,-23.486111,-46.366721,37.766919,547.608602
7,FC_3204_sao_paulo_SP,3204,sao paulo,SP,1415,-23.595499,-46.559727,23.732226,569.473538
8,FC_4160_sao_paulo_SP,4160,sao paulo,SP,1210,-23.625475,-46.612000,14.520240,666.587111
9,FC_13232_campo_limpo_paulista_SP,13232,campo limpo paulista,SP,1177,-23.211746,-46.762875,19.436066,508.751857



=== Demand and Capacity Planning Check ===
Total demand in selected customer zones: 51291
Total historical orders in selected seller centers: 23896

=== Step 6 Final Result ===
Saved customer demand zones to:
/Users/mac/Desktop/portfolio2_logistics_optimization/data/processed/customer_demand_zones.csv
Saved seller fulfillment centers to:
/Users/mac/Desktop/portfolio2_logistics_optimization/data/processed/seller_fulfillment_centers.csv


# Step 7: Build Distance-Cost Matrix and Fulfillment Center Capacity

This step creates the customer-center distance-cost matrix and constructs fulfillment center capacity for the optimization model.

In [5]:
# Step 7: Build distance-cost matrix and fulfillment center capacity

from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

# Set project directories
project_dir = Path("/Users/mac/Desktop/portfolio2_logistics_optimization")
data_processed_dir = project_dir / "data" / "processed"

# Load customer demand zones and seller fulfillment centers
customer_zones = pd.read_csv(data_processed_dir / "customer_demand_zones.csv")
seller_centers = pd.read_csv(data_processed_dir / "seller_fulfillment_centers.csv")

print("=== Input Table Shapes ===")
print("customer_zones:", customer_zones.shape)
print("seller_centers:", seller_centers.shape)

# Calculate total selected demand and historical seller order volume
total_demand = customer_zones["demand"].sum()
total_historical_orders = seller_centers["historical_orders"].sum()

print("\n=== Demand and Historical Seller Volume ===")
print("Total selected customer demand:", total_demand)
print("Total selected seller historical orders:", total_historical_orders)

# Construct fulfillment center capacity
# Capacity is allocated proportionally to each center's historical order volume.
# A capacity buffer is added to make the optimization model feasible.
capacity_buffer = 1.10
target_total_capacity = int(np.ceil(total_demand * capacity_buffer))

seller_centers["warehouse_capacity"] = (
    seller_centers["historical_orders"] / total_historical_orders * target_total_capacity
).round().astype(int)

# Adjust rounding difference to make sure total capacity is at least target_total_capacity
capacity_gap = target_total_capacity - seller_centers["warehouse_capacity"].sum()

if capacity_gap != 0:
    largest_center_index = seller_centers["warehouse_capacity"].idxmax()
    seller_centers.loc[largest_center_index, "warehouse_capacity"] += capacity_gap

seller_centers["capacity_utilization_if_proportional"] = (
    seller_centers["historical_orders"] / seller_centers["warehouse_capacity"]
)

print("\n=== Fulfillment Center Capacity Summary ===")
print("Target total capacity:", target_total_capacity)
print("Actual total capacity:", seller_centers["warehouse_capacity"].sum())
print("Capacity buffer:", capacity_buffer)

display(
    seller_centers[
        [
            "seller_center_id",
            "seller_city",
            "seller_state",
            "historical_orders",
            "warehouse_capacity",
            "capacity_utilization_if_proportional"
        ]
    ]
)

# Define haversine distance function
# This calculates approximate distance between two latitude-longitude points on Earth.
def haversine_distance_km(lat1, lon1, lat2, lon2):
    earth_radius_km = 6371
    
    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)
    
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    
    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2) ** 2
    )
    
    c = 2 * np.arcsin(np.sqrt(a))
    
    return earth_radius_km * c

# Create all customer-zone and fulfillment-center combinations
customer_zones["key"] = 1
seller_centers["key"] = 1

distance_cost_matrix = customer_zones.merge(
    seller_centers,
    on="key",
    how="inner"
).drop(columns=["key"])

# Calculate distance between each customer zone and each fulfillment center
distance_cost_matrix["distance_km"] = haversine_distance_km(
    distance_cost_matrix["seller_latitude"],
    distance_cost_matrix["seller_longitude"],
    distance_cost_matrix["customer_latitude"],
    distance_cost_matrix["customer_longitude"]
)

# Define transportation cost assumption
# cost_per_km_per_order is a simplified cost unit for optimization demonstration.
cost_per_km_per_order = 1.0

distance_cost_matrix["cost_per_km_per_order"] = cost_per_km_per_order
distance_cost_matrix["transportation_cost"] = (
    distance_cost_matrix["distance_km"]
    * distance_cost_matrix["demand"]
    * distance_cost_matrix["cost_per_km_per_order"]
)

print("\n=== Distance-Cost Matrix Shape ===")
print("distance_cost_matrix:", distance_cost_matrix.shape)

print("\n=== Distance Summary ===")
distance_summary = distance_cost_matrix["distance_km"].describe().reset_index()
distance_summary.columns = ["metric", "value"]
display(distance_summary)

print("\n=== Transportation Cost Summary ===")
cost_summary = distance_cost_matrix["transportation_cost"].describe().reset_index()
cost_summary.columns = ["metric", "value"]
display(cost_summary)

print("\n=== Sample Distance-Cost Matrix Rows ===")
sample_columns = [
    "customer_zone_id",
    "customer_city",
    "customer_state",
    "demand",
    "seller_center_id",
    "seller_city",
    "seller_state",
    "warehouse_capacity",
    "distance_km",
    "transportation_cost"
]

display(distance_cost_matrix[sample_columns].head(10))

# Remove temporary key columns from saved customer and seller tables if they exist
customer_zones = customer_zones.drop(columns=["key"], errors="ignore")
seller_centers = seller_centers.drop(columns=["key"], errors="ignore")

# Save processed tables
customer_output_path = data_processed_dir / "customer_demand_zones_with_capacity_check.csv"
seller_output_path = data_processed_dir / "seller_fulfillment_centers_with_capacity.csv"
matrix_output_path = data_processed_dir / "distance_cost_matrix.csv"

customer_zones.to_csv(customer_output_path, index=False)
seller_centers.to_csv(seller_output_path, index=False)
distance_cost_matrix.to_csv(matrix_output_path, index=False)

print("\n=== Step 7 Final Result ===")
print("Saved customer demand zones to:")
print(customer_output_path)
print("Saved seller fulfillment centers with capacity to:")
print(seller_output_path)
print("Saved distance-cost matrix to:")
print(matrix_output_path)

=== Input Table Shapes ===
customer_zones: (30, 8)
seller_centers: (12, 9)

=== Demand and Historical Seller Volume ===
Total selected customer demand: 51291
Total selected seller historical orders: 23896

=== Fulfillment Center Capacity Summary ===
Target total capacity: 56421
Actual total capacity: 56421
Capacity buffer: 1.1


,seller_center_id,seller_city,seller_state,historical_orders,warehouse_capacity,capacity_utilization_if_proportional
0,FC_14940_ibitinga_SP,ibitinga,SP,7595,17934,0.423497
1,FC_5849_sao_paulo_SP,sao paulo,SP,2004,4732,0.423500
2,FC_15025_sao_jose_do_rio_preto_SP,sao jose do rio preto,SP,1998,4717,0.423574
3,FC_9015_santo_andre_SP,santo andre,SP,1710,4037,0.423582
4,FC_13405_piracicaba_SP,piracicaba,SP,1562,3688,0.423536
5,FC_4782_sao_paulo_SP,sao paulo,SP,1485,3506,0.423560
6,FC_8577_itaquaquecetuba_SP,itaquaquecetuba,SP,1441,3402,0.423574
7,FC_3204_sao_paulo_SP,sao paulo,SP,1415,3341,0.423526
8,FC_4160_sao_paulo_SP,sao paulo,SP,1210,2857,0.423521
9,FC_13232_campo_limpo_paulista_SP,campo limpo paulista,SP,1177,2779,0.423534



=== Distance-Cost Matrix Shape ===
distance_cost_matrix: (360, 22)

=== Distance Summary ===


,metric,value
0,count,360.000000
1,mean,576.877201
2,std,654.561968
3,min,0.268307
4,25%,93.902909
5,50%,369.112756
6,75%,700.539838
7,max,2484.037270



=== Transportation Cost Summary ===


,metric,value
0,count,3.600000e+02
1,mean,6.748901e+05
2,std,9.159999e+05
3,min,2.334269e+02
4,25%,1.060564e+05
5,50%,2.794329e+05
6,75%,1.175531e+06
7,max,7.261746e+06



=== Sample Distance-Cost Matrix Rows ===


,customer_zone_id,customer_city,customer_state,demand,seller_center_id,seller_city,seller_state,warehouse_capacity,distance_km,transportation_cost
0,sao_paulo_SP,sao paulo,SP,17359,FC_14940_ibitinga_SP,ibitinga,SP,17934,302.447407,5.250185e+06
1,sao_paulo_SP,sao paulo,SP,17359,FC_5849_sao_paulo_SP,sao paulo,SP,4732,15.300012,2.655929e+05
2,sao_paulo_SP,sao paulo,SP,17359,FC_15025_sao_jose_do_rio_preto_SP,sao jose do rio preto,SP,4717,418.327458,7.261746e+06
3,sao_paulo_SP,sao paulo,SP,17359,FC_9015_santo_andre_SP,santo andre,SP,4037,14.886074,2.584074e+05
4,sao_paulo_SP,sao paulo,SP,17359,FC_13405_piracicaba_SP,piracicaba,SP,3688,142.560240,2.474703e+06
5,sao_paulo_SP,sao paulo,SP,17359,FC_4782_sao_paulo_SP,sao paulo,SP,3506,15.031276,2.609279e+05
6,sao_paulo_SP,sao paulo,SP,17359,FC_8577_itaquaquecetuba_SP,itaquaquecetuba,SP,3402,28.861617,5.010088e+05
7,sao_paulo_SP,sao paulo,SP,17359,FC_3204_sao_paulo_SP,sao paulo,SP,3341,7.996017,1.388029e+05
8,sao_paulo_SP,sao paulo,SP,17359,FC_4160_sao_paulo_SP,sao paulo,SP,2857,6.352208,1.102680e+05
9,sao_paulo_SP,sao paulo,SP,17359,FC_13232_campo_limpo_paulista_SP,campo limpo paulista,SP,2779,42.164287,7.319299e+05



=== Step 7 Final Result ===
Saved customer demand zones to:
/Users/mac/Desktop/portfolio2_logistics_optimization/data/processed/customer_demand_zones_with_capacity_check.csv
Saved seller fulfillment centers with capacity to:
/Users/mac/Desktop/portfolio2_logistics_optimization/data/processed/seller_fulfillment_centers_with_capacity.csv
Saved distance-cost matrix to:
/Users/mac/Desktop/portfolio2_logistics_optimization/data/processed/distance_cost_matrix.csv
